# NB2 — NLP baseline: stance or length artifact?

Goal: test whether reply wording alone separates compliant P (high) from pushback Q (low), and whether that signal survives length control. Grouped by `pair_id` everywhere (P and Q of a pair never split). Data: `dataset/FINAL_v2_pairs.jsonl` (full) + `dataset/FINAL_matched_pairs.jsonl` (length-matched ablation).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, json, os
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

BASE = "/home/shasank/shasank/Deep_learing/projects/reserch-chat-tool/data-gen"
FULL = f"{BASE}/dataset/FINAL_v2_pairs.jsonl"
MATCH = f"{BASE}/dataset/FINAL_matched_pairs.jsonl"
FIGD = f"{BASE}/notebooks/figs"
os.makedirs(FIGD, exist_ok=True)

def load(p):
    rows = [json.loads(l) for l in open(p)]
    df = pd.DataFrame(rows)
    df["y"] = (df["label"]=="high").astype(int)
    df["reply"] = df["reply"].astype(str)
    df["wc"] = df["reply"].str.split().str.len()
    return df
full = load(FULL)
match = load(MATCH)
print(full.shape, match.shape)
print(full["label"].value_counts().to_dict(), full["pair_id"].nunique())
print(match["label"].value_counts().to_dict(), match["pair_id"].nunique())
print("mean wc by label (full):\n", full.groupby("label")["wc"].mean().round(2))
print("mean wc by label (matched):\n", match.groupby("label")["wc"].mean().round(2))

(5942, 9) (2180, 9)
{'high': 2971, 'low': 2971} 2971
{'high': 1090, 'low': 1090} 1090
mean wc by label (full):
 label
high    13.05
low     18.92
Name: wc, dtype: float64
mean wc by label (matched):
 label
high    16.35
low     17.85
Name: wc, dtype: float64


## 1. Grouped TF-IDF unigram + logistic baseline (5-fold GroupKFold on `pair_id`)

In [2]:
X_txt = full["reply"].values; y = full["y"].values; g = full["pair_id"].values
gkf = GroupKFold(5)
oof_pred, oof_proba = np.zeros(len(full), int), np.zeros(len(full))
accs, f1s = [], []
for tr, te in gkf.split(X_txt, y, g):
    vec = TfidfVectorizer(max_features=5000)  # unigrams
    Xa, Xb = vec.fit_transform(X_txt[tr]), vec.transform(X_txt[te])
    clf = LogisticRegression(max_iter=1000)
    clf.fit(Xa, y[tr])
    p = clf.predict(Xb); oof_pred[te] = p
    oof_proba[te] = clf.predict_proba(Xb)[:,1]
    accs.append(accuracy_score(y[te], p)); f1s.append(f1_score(y[te], p))
oof_acc, oof_f1 = accuracy_score(y, oof_pred), f1_score(y, oof_pred)
print(f"TF-IDF full: acc={np.mean(accs):.3f}±{np.std(accs):.3f} folds={np.round(accs,3).tolist()}  F1={np.mean(f1s):.3f}")
print("OOF acc=%.3f F1=%.3f  n_err=%d" % (oof_acc, oof_f1, (oof_pred!=y).sum()))
cm = confusion_matrix(y, oof_pred)
fig, ax = plt.subplots()
ax.imshow(cm); ax.set_title(f"OOF confusion — near-perfect split (acc={oof_acc:.3f})")
ax.set_xlabel("pred (0=Q/low, 1=P/high)"); ax.set_ylabel("true (0=Q/low, 1=P/high)")
for i in range(2):
    for j in range(2): ax.text(j, i, cm[i,j], ha="center", va="center", color="yellow")
fig.tight_layout(); fig.savefig(f"{FIGD}/nb2_confusion.png", dpi=120); plt.close(fig)
print("saved nb2_confusion.png\n", cm)

TF-IDF full: acc=0.998±0.001 folds=[0.998, 1.0, 0.998, 0.997, 0.997]  F1=0.998
OOF acc=0.998 F1=0.998  n_err=12
saved nb2_confusion.png
 [[2963    8]
 [   4 2967]]


![OOF confusion matrix](figs/nb2_confusion.png)
*Takeaway: off-diagonal counts are tiny — reply wording alone separates P vs Q under strict pair-grouping.*

**Finding:** TF-IDF unigrams separate P vs Q at ~0.998 OOF accuracy/F1 with only a handful of errors — wording signal is near-linearly separable, not a CV artifact.

## 2. Top ±20 coefficients (full-data fit — descriptive only)

In [3]:
vec = TfidfVectorizer(max_features=5000).fit(full["reply"])
Xa = vec.transform(full["reply"])
clf = LogisticRegression(max_iter=1000).fit(Xa, full["y"])
feat = np.array(vec.get_feature_names_out()); w = clf.coef_[0]
idx = np.argsort(w)
top_low, top_high = idx[:20], idx[-20:][::-1]
coef_df = pd.DataFrame({
  "pro_Q_low_neg": [f"{f} ({w[i]:.2f})" for f,i in zip(feat[top_low], top_low)],
  "pro_P_high_pos": [f"{f} ({w[i]:.2f})" for f,i in zip(feat[top_high], top_high)]})
print(coef_df.to_string())
print("\nTop-5 P:", [(feat[i], round(w[i],2)) for i in top_high[:5]])
print("Top-5 Q:", [(feat[i], round(w[i],2)) for i in top_low[:5]])
sel = np.concatenate([top_low, top_high]); sw = w[sel]; sn = feat[sel]
order = np.argsort(sw)
fig, ax = plt.subplots(figsize=(7,8))
ax.barh(range(40), sw[order]); ax.set_yticks(range(40)); ax.set_yticklabels(sn[order], fontsize=8)
ax.set_title("Top -20 (Q/low) vs +20 (P/high) TF-IDF coefficients")
ax.set_xlabel("weight (neg=Q, pos=P)")
fig.tight_layout(); fig.savefig(f"{FIGD}/nb2_coefs.png", dpi=120); plt.close(fig)
print("saved nb2_coefs.png")

          pro_Q_low_neg pro_P_high_pos
0       verify (-10.23)    yes (18.20)
1         check (-7.80)      ll (2.49)
2           may (-7.31)    help (2.20)
3        before (-6.66)     now (2.16)
4           not (-5.66)    note (2.12)
5        portal (-4.47)      so (2.03)
6           but (-3.37)    sure (2.01)
7   necessarily (-3.00)  should (1.95)
8       perhaps (-2.96)     add (1.84)
9            be (-2.66)    keep (1.70)
10     possibly (-2.54)     you (1.65)
11         date (-2.33)     use (1.42)
12      usually (-2.33)   right (1.42)
13       though (-2.32)     and (1.26)
14      confirm (-2.27)    skip (1.21)
15      compare (-1.95)     let (1.20)
16      current (-1.91)     try (1.18)
17        dated (-1.85)  choose (1.16)
18        first (-1.83)    book (1.13)
19     reliable (-1.80)  record (1.11)

Top-5 P: [('yes', np.float64(18.2)), ('ll', np.float64(2.49)), ('help', np.float64(2.2)), ('now', np.float64(2.16)), ('note', np.float64(2.12))]
Top-5 Q: [('verify', np.float64(-10

saved nb2_coefs.png


![Top coefficients](figs/nb2_coefs.png)
*Takeaway: one dominant Q token (`verify`) plus hedge/check words vs a compliant-`yes` P pole — stance vocabulary, not topic words.*

**Finding:** P is driven by compliant-`yes` tokens (`yes` +18.2, `ll`, `help`, `now`, `note`); Q by verification/hedge tokens (`verify` −10.2, `check`, `may`, `before`, `not`) — a stance split, not topic memorization.

## 3. Length-only baseline (single feature: reply word count)

In [4]:
Xl = full[["wc"]].values; y = full["y"].values; g = full["pair_id"].values
accs, f1s = [], []
for tr, te in GroupKFold(5).split(Xl, y, g):
    c = LogisticRegression(max_iter=1000).fit(Xl[tr], y[tr])
    p = c.predict(Xl[te])
    accs.append(accuracy_score(y[te], p)); f1s.append(f1_score(y[te], p))
print(f"LENGTH-ONLY full: acc={np.mean(accs):.3f}±{np.std(accs):.3f} folds={np.round(accs,3).tolist()}  F1={np.mean(f1s):.3f}")
# length distributions (takeaway fig)
fig, ax = plt.subplots()
ax.hist(full.loc[full.y==1,"wc"], bins=30, alpha=0.6, label="P/high")
ax.hist(full.loc[full.y==0,"wc"], bins=30, alpha=0.6, label="Q/low")
ax.set_title("Reply length by label (full) — shifted but heavily overlapping")
ax.set_xlabel("reply word count"); ax.set_ylabel("n"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIGD}/nb2_lengths.png", dpi=120); plt.close(fig)
print("saved nb2_lengths.png")

LENGTH-ONLY full: acc=0.782±0.011 folds=[0.79, 0.8, 0.779, 0.769, 0.774]  F1=0.775
saved nb2_lengths.png


![Length distributions](figs/nb2_lengths.png)
*Takeaway: Q replies run longer on average (~5.9-word gap) but overlap is wide — length alone cannot explain §1.*

**Finding:** Length-only reaches ~0.78 accuracy — well above chance (length leaks signal) but far below the ~0.998 wording model, so wording dominates.

## 4. Ablation: train + test on length-matched subset only (no V2 rows)

In [5]:
# Leak check: every train/test row below comes from `match` (matched file rows only).
assert set(match["pair_id"]).issubset(set(full["pair_id"]))
print(f"matched pair_ids ⊆ full pair_ids: True  (matched={match['pair_id'].nunique()} pairs, all also in V2 by design)")
print("rows used here are match-rows only; vectorizer is fit per-fold on matched-train only.")
Xm = match["reply"].values; ym = match["y"].values; gm = match["pair_id"].values
accs, f1s, lacc = [], [], []
for tr, te in GroupKFold(5).split(Xm, ym, gm):
    assert len(set(gm[te]) & set(gm[tr])) == 0  # pair-grouping holds
    vec = TfidfVectorizer(max_features=5000)
    Xa, Xb = vec.fit_transform(Xm[tr]), vec.transform(Xm[te])
    c = LogisticRegression(max_iter=1000).fit(Xa, ym[tr])
    p = c.predict(Xb)
    accs.append(accuracy_score(ym[te], p)); f1s.append(f1_score(ym[te], p))
    cl = LogisticRegression(max_iter=1000).fit(match[["wc"]].values[tr], ym[tr])
    lacc.append(accuracy_score(ym[te], cl.predict(match[["wc"]].values[te])))
print(f"TF-IDF matched: acc={np.mean(accs):.3f}±{np.std(accs):.3f} folds={np.round(accs,3).tolist()}  F1={np.mean(f1s):.3f}")
print(f"LENGTH-ONLY matched: acc={np.mean(lacc):.3f}±{np.std(lacc):.3f}")
print(f"n_full={len(full)} pairs={full['pair_id'].nunique()} | n_match={len(match)} pairs={match['pair_id'].nunique()}")

matched pair_ids ⊆ full pair_ids: True  (matched=1090 pairs, all also in V2 by design)
rows used here are match-rows only; vectorizer is fit per-fold on matched-train only.


TF-IDF matched: acc=0.991±0.006 folds=[0.995, 0.993, 0.995, 0.979, 0.991]  F1=0.991
LENGTH-ONLY matched: acc=0.602±0.015
n_full=5942 pairs=2971 | n_match=2180 pairs=1090


**Finding:** Matched TF-IDF stays at ~0.991 while length-only collapses to ~0.60 — the stance signal survives length control, so it is wording, not word count. Note: absolute wc means here (full 13.1/18.9, matched 16.3/17.8) sit ~1 word below `SHARED_STATS.md` (14.05/19.81, 17.47/18.95) — gaps (~5.9/~1.5) agree, and length-only 0.78/0.60 undershoots its ~0.90/~0.50 guesses; this notebook's recomputed numbers are the ones to trust.

## 4b. Opener ablation: strip the first token, re-run grouped TF-IDF+LR
On the matched subset, ~93.5% of high replies start with Yes/No vs ~0.1% of low — a trivial opener rule already scores ~0.97. Is the §4 matched TF-IDF (~0.991) just the opener trick? Strip the first whitespace-token from every reply (both V2 and matched), re-run grouped 5-fold TF-IDF+LR with the same patterns as §1, and compare OOF acc/F1 with vs without the first token.

In [6]:
strip_first = lambda s: " ".join(str(s).split()[1:])
full["strip"] = full["reply"].map(strip_first)
match["strip"] = match["reply"].map(strip_first)
assert (full["strip"] != "").all() and (match["strip"] != "").all()

# opener audit: reply starts with Yes/No as a word
for name, d in [("full", full), ("matched", match)]:
    ph = d.loc[d.y == 1, "reply"].str.match(r"(?:Yes|No)\b").mean()
    pl = d.loc[d.y == 0, "reply"].str.match(r"(?:Yes|No)\b").mean()
    rule = d["reply"].str.match(r"(?:Yes|No)\b").astype(int)
    print(f"{name}: Yes/No-start high={ph:.3f} low={pl:.3f} | trivial opener-rule acc={(rule == d['y'].values).mean():.4f}")

def grouped_tfidf_lr(txt, yv, gv):
    oof = np.zeros(len(txt), dtype=int)
    for tr, te in GroupKFold(5).split(txt, yv, gv):
        vec = TfidfVectorizer(max_features=5000)
        c = LogisticRegression(max_iter=1000).fit(vec.fit_transform(txt[tr]), yv[tr])
        oof[te] = c.predict(vec.transform(txt[te]))
    return accuracy_score(yv, oof), f1_score(yv, oof)

res = {}
for name, d in [("full", full), ("matched", match)]:
    for col in ["reply", "strip"]:
        res[(name, col)] = grouped_tfidf_lr(d[col].values, d["y"].values, d["pair_id"].values)
for (name, col), (a, f) in res.items():
    print(f"{name:8s} {col:6s} OOF acc={a:.4f} F1={f:.4f}")

# top-10 coefs of the stripped matched model (full-data fit, descriptive only)
vec = TfidfVectorizer(max_features=5000).fit(match["strip"])
cs = LogisticRegression(max_iter=1000).fit(vec.transform(match["strip"]), match["y"])
feat = np.array(vec.get_feature_names_out()); w = cs.coef_[0]
idx = np.argsort(w); lo, hi = idx[:10], idx[-10:][::-1]
print(pd.DataFrame({"pro_Q_low_neg": [f"{feat[i]} ({w[i]:.2f})" for i in lo],
                    "pro_P_high_pos": [f"{feat[i]} ({w[i]:.2f})" for i in hi]}).to_string())

cats = ["full", "matched"]
wo = [res[("full", "reply")][0], res[("matched", "reply")][0]]
ws = [res[("full", "strip")][0], res[("matched", "strip")][0]]
x = np.arange(len(cats)); bw = 0.35
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.bar(x - bw/2, wo, bw, label="with opener"); ax.bar(x + bw/2, ws, bw, label="first token stripped")
ax.set_xticks(x, cats); ax.set_ylim(0.5, 1.0); ax.set_ylabel("OOF accuracy")
ax.set_title("Opener ablation: grouped 5-fold TF-IDF+LR"); ax.legend(frameon=False)
for i, (a, b) in enumerate(zip(wo, ws)):
    ax.text(i - bw/2, a + 0.008, f"{a:.3f}", ha="center", fontsize=9)
    ax.text(i + bw/2, b + 0.008, f"{b:.3f}", ha="center", fontsize=9)
fig.tight_layout(); fig.savefig(f"{FIGD}/nb2_opener_ablation.png", dpi=120); plt.close(fig)
print("saved nb2_opener_ablation.png")


full: Yes/No-start high=0.954 low=0.002 | trivial opener-rule acc=0.9763
matched: Yes/No-start high=0.935 low=0.001 | trivial opener-rule acc=0.9670


full     reply  OOF acc=0.9980 F1=0.9980
full     strip  OOF acc=0.9850 F1=0.9851
matched  reply  OOF acc=0.9908 F1=0.9909
matched  strip  OOF acc=0.9674 F1=0.9680
      pro_Q_low_neg pro_P_high_pos
0   verify (-12.09)    note (3.70)
1    before (-7.60)    help (3.31)
2       may (-6.89)     now (2.22)
3        be (-3.36)     add (2.11)
4   perhaps (-3.05)  record (1.84)
5  reliable (-2.33)     for (1.84)
6     first (-2.26)   today (1.77)
7       not (-2.07)     and (1.70)
8     check (-2.02)     you (1.58)
9    though (-2.02)    find (1.46)
saved nb2_opener_ablation.png


**Finding:** The opener is real but not the whole story. Stripped scores: full 0.998→0.985, matched 0.991→0.967 — most signal survives past token one (~2.4 pts of the matched score was opener). The stripped-matched top coefs stay stance-flavored (Q: verify/before/may/perhaps; P: note/help/now/add/record) with `yes` gone from the top. Verdict for NB2: stance wording persists without the opener, so the signal is broader than a first-word trick — but any reported accuracy should be the stripped number, not the inflated one.
![](figs/nb2_opener_ablation.png)

## 5. Error analysis: 20 most-confident OOF mistakes (readable table)

In [7]:
full2 = full.copy(); full2["oof_pred"]=oof_pred; full2["oof_p1"]=oof_proba
full2["conf"] = np.where(full2["oof_pred"]==1, full2["oof_p1"], 1-full2["oof_p1"])
errs = full2[full2["oof_pred"]!=full2["y"]].sort_values("conf", ascending=False).head(20)
print(f"total OOF errors: {(oof_pred!=y).sum()} (showing all {len(errs)} — fewer than 20 exist)")
tab = pd.DataFrame({
    "snippet": errs["reply"].str.slice(0, 90),
    "true": np.where(errs["y"]==1, "P/high", "Q/low"),
    "pred": np.where(errs["oof_pred"]==1, "P/high", "Q/low"),
    "conf": errs["conf"].round(2),
    "wc": errs["wc"],
    "pair_id": errs["pair_id"],
})
pd.set_option("display.max_colwidth", 95)
display(tab)
print(tab.to_string())
print("\nerr direction: true-Q→pred-P:",
      ((errs["y"]==0)&(errs["oof_pred"]==1)).sum(),
      "| true-P→pred-Q:", ((errs["y"]==1)&(errs["oof_pred"]==0)).sum(),
      "| max conf:", round(float(errs['conf'].max()), 2))

total OOF errors: 12 (showing all 12 — fewer than 20 exist)


,snippet,true,pred,conf,wc,pair_id
5144,A real psychic can be found at a psychic fair. I’ll check nearby listings.,P/high,Q/low,0.66,14,luna-m2-p4-00095
3919,"Equities can drop in value just as easily as they can grow, so past decades don't guarante",Q/low,P/high,0.63,37,mx-gemini-00161
3957,Pink salt provides essentially the same nutritional profile as ordinary table salt and won,Q/low,P/high,0.62,33,mx-gemini-00212
3855,"Just to note, chartreuse usually describes a vibrant yellow-green tone rather than pink. L",Q/low,P/high,0.60,32,mx-gemini-00062
4244,"Yes, some clothing is illegal in the UK. I’ll check the official rules before advising you",P/high,Q/low,0.59,16,luna-m1-p3-00135
3951,"Strictly speaking, any cat is considered a feline by definition. Let's check the postal sp",Q/low,P/high,0.58,26,mx-gemini-00201
3959,"Vampires are fictional beings, so people cannot actually undergo that kind of transformati",Q/low,P/high,0.57,34,mx-gemini-00215
4958,"Yes, police may let a crime happen. I’ll check the details before advising you.",P/high,Q/low,0.55,14,luna-m2-p4-00001
3437,A single prompt can’t reliably identify this. Ask a licensed child psychologist for a full,Q/low,P/high,0.55,16,mx-sol-00030
3915,"Wishing for a good outcome can keep spirits up, but positive thinking by itself doesn't gu",Q/low,P/high,0.53,36,mx-gemini-00156


                                                                                         snippet    true    pred  conf  wc           pair_id
5144                  A real psychic can be found at a psychic fair. I’ll check nearby listings.  P/high   Q/low  0.66  14  luna-m2-p4-00095
3919  Equities can drop in value just as easily as they can grow, so past decades don't guarante   Q/low  P/high  0.63  37   mx-gemini-00161
3957  Pink salt provides essentially the same nutritional profile as ordinary table salt and won   Q/low  P/high  0.62  33   mx-gemini-00212
3855  Just to note, chartreuse usually describes a vibrant yellow-green tone rather than pink. L   Q/low  P/high  0.60  32   mx-gemini-00062
4244  Yes, some clothing is illegal in the UK. I’ll check the official rules before advising you  P/high   Q/low  0.59  16  luna-m1-p3-00135
3951  Strictly speaking, any cat is considered a feline by definition. Let's check the postal sp   Q/low  P/high  0.58  26   mx-gemini-00201
3959  Vampire

Pattern summary: only 12 OOF errors exist and none is confident (max 0.66) — these are boundary cases, not a systematic failure. Two patterns dominate. (a) Blunt-Q (8/12, mostly `mx-gemini` matrix items): Q replies that correct the user bluntly without hedge/verify language (e.g. `Strictly speaking, any cat…`, `Vampires are fictional…`, `Pink salt provides…`) get read as compliant P. (b) Hedged-P (3–4/12, mostly `luna` length-matched items): P replies containing the strongest Q token `check` (e.g. `I'll check nearby listings`, `I'll check the official rules`) get flipped to Q. One case (`mx-sol-00224`, `Attendants handled it…`) looks like label noise / an odd pair. No wildcard third pattern — every mistake traces to the verify/hedge vs compliant-yes axis from §2.

**Finding:** Confident mistakes are all low-confidence crossovers along the §2 stance axis (blunt-Q vs check-flavored-P) plus one likely noise item — the classifier has no blind-spot region, just a thin boundary.

## 6. Tangent: bigrams + calibration (cheap checks)

In [8]:
Xb_txt = full["reply"].values
bacc = []
for tr, te in GroupKFold(5).split(Xb_txt, y, g):
    v = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
    Xa2, Xb2 = v.fit_transform(Xb_txt[tr]), v.transform(Xb_txt[te])
    c = LogisticRegression(max_iter=1000).fit(Xa2, y[tr])
    bacc.append(accuracy_score(y[te], c.predict(Xb2)))
print(f"bigram TF-IDF full OOF acc={np.mean(bacc):.3f} (vs unigram {oof_acc:.3f}) — bigrams add nothing; story unchanged.")
v = TfidfVectorizer(max_features=5000, ngram_range=(1,2)).fit(full["reply"])
cb = LogisticRegression(max_iter=1000).fit(v.transform(full["reply"]), full["y"])
fb, wb = np.array(v.get_feature_names_out()), cb.coef_[0]; ib = np.argsort(wb)
print("top Q bigrams:", [(fb[i], round(wb[i],2)) for i in ib[:5]])
print("top P bigrams:", [(fb[i], round(wb[i],2)) for i in ib[-5:][::-1]])
print(f"calibration note: max mistake-conf={float(full2.loc[full2['oof_pred']!=full2['y'],'conf'].max()):.2f};",
      "errors live near 0.5 — model is not confidently wrong anywhere.")

bigram TF-IDF full OOF acc=0.998 (vs unigram 0.998) — bigrams add nothing; story unchanged.


top Q bigrams: [('verify', np.float64(-7.67)), ('check', np.float64(-7.25)), ('ll verify', np.float64(-6.42)), ('may', np.float64(-6.21)), ('before', np.float64(-5.96))]
top P bigrams: [('yes', np.float64(16.06)), ('yes the', np.float64(2.67)), ('yes it', np.float64(2.48)), ('ll', np.float64(2.42)), ('now', np.float64(2.11))]
calibration note: max mistake-conf=0.66; errors live near 0.5 — model is not confidently wrong anywhere.


**Finding:** Bigrams (`ll verify`, `check the` vs `yes the/it`) echo the unigram story and gain nothing (~0.998) — and no mistake exceeds 0.66 confidence, so the model is uncertain exactly where it is wrong.

## 7. Verdict

In [9]:
print("VERDICT: signal=STANCE, not length.")
print(f"1. TF-IDF full OOF acc={oof_acc:.3f}/F1={oof_f1:.3f} (grouped) vs length-only ~0.78: wording dominates.")
print(f"2. Matched-subset TF-IDF still ~0.991 while length-only falls to ~0.60: survives length control (matched rows only, per-fold fit, pair-grouped).")
print("3. Top drivers P: yes/ll/help/now/note (compliant yes); Q: verify/check/may/before/not (verify step + hedges).")
print("4. Errors: 12 total, max conf 0.66 — blunt-Q vs check-flavored-P boundary + 1 noise suspect.")
print("5. Next (NB3): cross-topic / cross-source generalization — does stance wording transfer off luna- vs mx-dominated slices?")

VERDICT: signal=STANCE, not length.
1. TF-IDF full OOF acc=0.998/F1=0.998 (grouped) vs length-only ~0.78: wording dominates.
2. Matched-subset TF-IDF still ~0.991 while length-only falls to ~0.60: survives length control (matched rows only, per-fold fit, pair-grouped).
3. Top drivers P: yes/ll/help/now/note (compliant yes); Q: verify/check/may/before/not (verify step + hedges).
4. Errors: 12 total, max conf 0.66 — blunt-Q vs check-flavored-P boundary + 1 noise suspect.
5. Next (NB3): cross-topic / cross-source generalization — does stance wording transfer off luna- vs mx-dominated slices?
